# 📦 BNNR — Bring Your Own Data

This notebook shows how to use BNNR with **your own** classification data (v0.1.2: single-label and multi-label).

**Path:** **Classification** — `ImageFolder` layout + BNNR Python API (full control). For multi-label, see `examples/multilabel/bnnr_multilabel_demo.ipynb` and `docs/golden_path.md`.

⏱ **Runtime:** Depends on your data size. The examples below use small synthetic data for demonstration.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mateuszwalo/BNNR/blob/main/examples/bnnr_custom_data.ipynb)

## 0. Installation

In [ ]:
%pip install -q bnnr

## 1. Upload Your Data to Colab

### Option A: Google Drive (recommended for large datasets)
```python
from google.colab import drive
drive.mount('/content/drive')
# Your data is now at /content/drive/MyDrive/your_dataset/
```

### Option B: Direct upload (small datasets < 100 MB)
```python
from google.colab import files
uploaded = files.upload()  # Opens a file picker
# If it's a zip: !unzip uploaded_file.zip -d my_data/
```

### Option C: Download from URL
```python
!wget -q https://example.com/dataset.zip
!unzip -q dataset.zip -d my_data/
```

---

# Part A: Classification with Custom Data

## A1. Prepare Your Data

BNNR uses PyTorch's `ImageFolder` format. Organize your images like this:

```
my_data/
  train/
    class_a/
      img001.jpg
      img002.jpg
    class_b/
      img003.jpg
      ...
  val/
    class_a/
      img100.jpg
    class_b/
      img101.jpg
```

Each subfolder name = class name. Images can be any format (jpg, png, etc.).

**For this demo, we create a small synthetic dataset to show the workflow:**

In [ ]:
import os
import numpy as np
from PIL import Image

# Create synthetic ImageFolder data for demo
np.random.seed(42)
classes = ["cats", "dogs", "birds"]
for split in ["train", "val"]:
    n = 50 if split == "train" else 20
    for cls in classes:
        path = f"my_data/{split}/{cls}"
        os.makedirs(path, exist_ok=True)
        for i in range(n):
            img = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
            Image.fromarray(img).save(f"{path}/{i:04d}.png")

print("Synthetic ImageFolder dataset created:")
for split in ["train", "val"]:
    for cls in classes:
        n = len(os.listdir(f"my_data/{split}/{cls}"))
        print(f"  {split}/{cls}: {n} images")

## A2. Load Data + Define Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from bnnr import (
    BNNRConfig,
    BNNRTrainer,
    SimpleTorchAdapter,
    auto_select_augmentations,
)
from bnnr.icd import ICD, AICD
from bnnr.xai_cache import XAICache

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 3  # cats, dogs, birds


class IndexedDataset(Dataset):
    """Adds index to each sample for ICD/AICD caching."""
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        img, label = self.ds[idx]
        return img, label, idx


# IMPORTANT: NO Normalize()! BNNR augmentations need raw [0,1] tensors.
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder("my_data/train", transform=transform)
val_ds = datasets.ImageFolder("my_data/val", transform=transform)

train_loader = DataLoader(IndexedDataset(train_ds), batch_size=32, shuffle=True)
val_loader = DataLoader(IndexedDataset(val_ds), batch_size=32)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Classes: {train_ds.classes}")

# Simple model
class SimpleNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),  # target layer
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

    @property
    def target_layer(self):
        return self.features[8]


model = SimpleNet(num_classes=NUM_CLASSES)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## A3. Configure and Train

We use BNNR built-in augmentations + ICD/AICD to search for the best augmentation policy.

In [ ]:
adapter = SimpleTorchAdapter(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    target_layers=[model.target_layer],
    device=DEVICE,
)

# Precompute XAI cache for ICD/AICD
xai_cache = XAICache(cache_dir="reports/custom_cls/xai_cache")
xai_cache.precompute_cache(
    model=model, train_loader=train_loader,
    target_layers=[model.target_layer],
    n_samples=len(train_ds), method="opticam",
)

# Build augmentations
augmentations = auto_select_augmentations()

# Add ICD and AICD
icd = ICD(
    model=model, target_layers=[model.target_layer],
    cache=xai_cache, probability=0.2, random_state=42,
)
icd.name = "icd"
augmentations.append(icd)

aicd = AICD(
    model=model, target_layers=[model.target_layer],
    cache=xai_cache, probability=0.2, random_state=43,
)
aicd.name = "aicd"
augmentations.append(aicd)

config = BNNRConfig(
    m_epochs=3,
    max_iterations=3,
    selection_metric="accuracy",
    device=DEVICE,
    xai_enabled=True,
    xai_method="opticam",
    report_dir="reports/custom_cls",
    event_log_enabled=True,
    seed=42,
)

print(f"{len(augmentations)} augmentation candidates")
print(f"Selection metric: {config.selection_metric}")



In [ ]:
trainer = BNNRTrainer(adapter, train_loader, val_loader, augmentations, config)
result = trainer.run()

print("\n" + "=" * 50)
print(f"Best path:    {result.best_path}")
print(f"Best metrics: {result.best_metrics}")
print(f"Selected:     {result.selected_augmentations}")
print("=" * 50)


---

## Tips & Best Practices

### General
- **Do NOT use `transforms.Normalize()`** in the data pipeline. BNNR augmentations need raw `[0, 1]` tensors. Use `BatchNorm` in the model instead.
- **Always include sample index** in the dataset output (3-tuple) — required for ICD/AICD caching.
- **Start small**: Use `max_iterations=3`, `m_epochs=3` for initial testing, then scale up.

### Classification
- Include **ICD + AICD** in your candidate pool — they use XAI to intelligently augment images.
- For small datasets (<5K images), aggressive augmentations help more.
- Use `auto_select_augmentations()` as a starting point, then add ICD/AICD.

### Dashboard (local only)
- Add `--with-dashboard` (CLI) to enable live monitoring locally.
- Replay after training: `bnnr dashboard serve --run-dir reports/your_run`

---

## Next Steps

- [Classification Demo](classification/bnnr_classification_demo.ipynb) — Full STL-10 demo with all augmentations
- [Multi-label Demo](multilabel/bnnr_multilabel_demo.ipynb) — Multi-label pipeline
- [Augmentations Guide](bnnr_augmentations_guide.ipynb) — Visual guide to every BNNR augmentation
- [Configuration Reference](../docs/configuration.md) — All BNNRConfig fields explained